# Automated Equity Valuation Model
### Discounted Cash Flow, Sensitivity Analysis, and Investment Memo

**Author:** Jason  ·  **Environment:** Google Colab  ·  **Example company:** Apple Inc. (AAPL)

---

## 1. Project Overview

This notebook takes a single public-company **ticker** and automatically produces a full, defensible equity valuation:

1. Pulls financial statements and market data from Yahoo Finance (`yfinance`).
2. Cleans the data and computes historical financial metrics.
3. Builds a **5-year unlevered Discounted Cash Flow (DCF)** model.
4. Estimates the discount rate with a **WACC / CAPM** build-up.
5. Computes a terminal value using **two independent methods** (Gordon Growth *and* an EV/EBITDA exit multiple), the way a real analyst triangulates.
6. Runs **bull / base / bear scenarios** and **two-variable sensitivity tables**.
7. Reverse-engineers the **growth the market is currently pricing in** (a reverse DCF).
8. Produces **presentation-quality charts**, a **football-field valuation range**, and a written **investment memo** with a final rating.

The goal is a model that is *honest* rather than rigged to a pretty answer. For mega-cap compounders like Apple, a plain perpetuity DCF often reads "overvalued," and the notebook explains *why* instead of hiding it.

## 2. Real-World Finance Use Case

The same machinery sits at the center of professional valuation work:

- **Equity research** analysts publish price targets built on exactly this DCF + scenario + sensitivity structure.
- **Investment banking** teams use DCFs (alongside comps and precedent transactions) in pitch books, fairness opinions, and M&A models.
- **Private equity** firms run unlevered FCF projections to set entry prices and underwrite returns.
- **Hedge funds and asset managers** use intrinsic-value estimates and reverse DCFs to decide whether the market's implied assumptions are beatable.

A DCF is never "the answer." It is a structured way to make assumptions explicit and stress-test them. That framing is the point of this project.

## 3. System Architecture

```
 ┌─────────────────┐   ┌──────────────────┐   ┌─────────────────────┐
 │ 1. Data          │  │ 2. Cleaning &     │  │ 3. Historical        │
 │    Collection    │→ │    Normalization  │→ │    Analysis / Metrics │
 │ (yfinance)       │  │ (robust lookups)  │  │ (growth, margins)     │
 └─────────────────┘   └──────────────────┘   └──────────┬──────────┘
                                                          │
 ┌─────────────────┐   ┌──────────────────┐   ┌──────────▼──────────┐
 │ 7. Sensitivity  │  │ 6. Scenarios      │  │ 4. Assumption Engine │
 │  & Reverse DCF  │← │  (Bull/Base/Bear) │← │ (defaults + overrides)│
 └────────┬────────┘   └────────┬─────────┘   └──────────┬──────────┘
          │                     │                        │
          │            ┌────────▼─────────┐   ┌──────────▼──────────┐
          │            │ 5b. WACC (CAPM)  │   │ 5a. DCF Engine       │
          │            │  discount rate   │→  │  UFCF → TV → EV →     │
          │            └──────────────────┘   │  Equity → $/share     │
          │                                    └──────────┬──────────┘
 ┌────────▼────────┐                                       │
 │ 8. Visualizations│  ┌──────────────────┐   ┌───────────▼─────────┐
 │  + Football Field│→ │ 9. Investment    │→  │ 10. Summary + Rating │
 └─────────────────┘   │    Memo          │   └─────────────────────┘
                       └──────────────────┘
```

**Design principle:** one engine, many calls. A single `run_dcf()` function is the core; scenarios, sensitivity, and the reverse DCF all just call it with modified assumptions. That keeps the logic auditable and is easy to defend in an interview.

In [ ]:
# =====================================================================
# CELL 2 — Install & Import Libraries
# =====================================================================
# In Colab, yfinance is usually present but often out of date. Upgrading
# avoids the most common "empty financials" bug. The line is quiet so it
# does not clutter the notebook. Comment it out if you are offline.
import sys, subprocess
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--upgrade", "yfinance"], check=False)
except Exception as e:
    print("pip install skipped:", e)

import warnings
warnings.filterwarnings("ignore")          # silence noisy yfinance/pandas warnings

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# seaborn is optional (used only for prettier heatmaps); degrade gracefully
try:
    import seaborn as sns
    sns.set_style("whitegrid")
    HAS_SNS = True
except Exception:
    HAS_SNS = False

try:
    import yfinance as yf
    HAS_YF = True
except Exception:
    HAS_YF = False
    print("yfinance not available — the notebook will run on bundled fallback data.")

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlesize": 11})

# House color palette (navy / green / red / grey) for a clean finance look
NAVY, GREEN, RED, GREY, GOLD = "#1f3b73", "#2a7a3b", "#b3401f", "#6b7280", "#b8860b"
print("Libraries loaded. yfinance available:", HAS_YF, "| seaborn:", HAS_SNS)

In [ ]:
# =====================================================================
# CELL 3 — CONTROL PANEL  (the only place you normally need to edit)
# =====================================================================
# Change the ticker and any assumptions here, then Runtime > Run all.

TICKER          = "AAPL"     # <-- change me (e.g. "MSFT", "NVDA", "KO", "JPM")
FORECAST_YEARS  = 5          # length of the explicit forecast window
FORCE_FALLBACK  = False      # True = run 100% offline on the bundled snapshot
RATING_BAND     = 0.15       # +/-15% deadband around price = "fairly valued"
BLEND_WEIGHT_GORDON = 0.50   # weight on Gordon Growth in the blended fair value

# ---- Assumption OVERRIDES -------------------------------------------------
# Leave empty {} to let the model derive defaults from the company's history.
# Uncomment / edit any line to override a single assumption. Rates are decimals
# (0.06 = 6%). EBIT margin and tax rate are also decimals.
USER_OVERRIDES = {
    # "rev_growth":        0.06,   # base-case annual revenue growth
    # "ebit_margin":       0.30,   # operating (EBIT) margin
    # "tax_rate":          0.16,   # effective tax rate
    # "da_pct_rev":        0.030,  # D&A as % of revenue
    # "capex_pct_rev":     0.028,  # CapEx as % of revenue
    # "nwc_pct_rev":       0.02,   # net working capital as % of revenue
    # "risk_free":         0.043,  # 10Y Treasury proxy
    # "market_risk_premium":0.047, # equity risk premium (Damodaran ~4.5-5%)
    # "cost_of_debt":      0.045,  # pre-tax cost of debt
    # "terminal_growth":   0.025,  # perpetuity growth (<= long-run GDP)
    # "exit_multiple":     18.0,   # EV/EBITDA for the exit-multiple TV method
}

# ---- Scenario deltas (applied on top of the base case) --------------------
SCENARIO_DELTAS = {
    "Bear": {"rev_growth": -0.03, "ebit_margin": -0.03,
             "terminal_growth": -0.010, "exit_multiple": -4.0, "wacc": +0.010},
    "Base": {"rev_growth":  0.00, "ebit_margin":  0.00,
             "terminal_growth":  0.000, "exit_multiple":  0.0, "wacc":  0.000},
    "Bull": {"rev_growth": +0.03, "ebit_margin": +0.03,
             "terminal_growth": +0.005, "exit_multiple": +4.0, "wacc": -0.010},
}
print(f"Configured for {TICKER} | {FORECAST_YEARS}-yr forecast | "
      f"force_fallback={FORCE_FALLBACK}")

In [ ]:
# =====================================================================
# CELL 4 — Data Collection Layer (functions + bundled fallback)
# =====================================================================
# All monetary values are normalized to USD MILLIONS, shares to MILLIONS,
# and price to actual dollars/share so that equity_value ($M) / shares (M)
# returns a clean $/share figure.

# ---- Bundled fallback (illustrative AAPL snapshot) ------------------------
# Used ONLY if the live pull fails or FORCE_FALLBACK is True. Clearly flagged
# in every downstream output so a reader never mistakes it for live data.
FALLBACK = {
    "ticker": "AAPL", "name": "Apple Inc. (fallback snapshot)",
    "price": 195.00, "shares_out": 15_204.0, "beta": 1.25,
    "market_cap": 195.00 * 15_204.0, "currency": "USD",
    "years":   [2021, 2022, 2023, 2024],
    "revenue": [365_817, 394_328, 383_285, 391_035],
    "ebit":    [108_949, 119_437, 114_301, 123_216],
    "net_income": [94_680, 99_803, 96_995, 93_736],
    "dep_amort": [11_284, 11_104, 11_519, 11_445],
    "capex":     [11_085, 10_708, 10_959, 9_447],
    "total_debt": 106_629.0, "cash": 65_171.0, "total_equity": 56_950.0,
    "effective_tax": 0.16, "is_fallback": True,
}

def get_row(df, candidates):
    """Find a statement row by trying several possible label names
    (exact match first, then case-insensitive substring). yfinance label
    names drift between versions, so this is what makes the pull robust."""
    if df is None or getattr(df, "empty", True):
        return None
    idx_map = {str(i).strip().lower(): i for i in df.index}
    for c in candidates:                       # exact
        if c.strip().lower() in idx_map:
            return df.loc[idx_map[c.strip().lower()]]
    for c in candidates:                       # substring
        cl = c.strip().lower()
        for k, orig in idx_map.items():
            if cl in k:
                return df.loc[orig]
    return None

def _latest(series, default=np.nan):
    if series is None:
        return default
    s = series.dropna()
    return float(s.iloc[0]) if len(s) else default

def fetch_financials(ticker, fallback=FALLBACK, force_fallback=False, retries=2):
    """Pull statements + market data from yfinance, normalize to $M, and
    return a single dict. Falls back to the bundled snapshot on any failure."""
    if force_fallback or not HAS_YF:
        fb = dict(fallback); fb["ticker"] = ticker if force_fallback else fb["ticker"]
        return fb

    last_err = None
    for attempt in range(retries + 1):
        try:
            tk = yf.Ticker(ticker)
            inc, bs, cf = tk.income_stmt, tk.balance_sheet, tk.cashflow

            rev_row = get_row(inc, ["Total Revenue", "Operating Revenue", "Revenue"])
            if rev_row is None or rev_row.dropna().empty:
                raise ValueError("income statement missing revenue")

            rev_s = rev_row.dropna()
            take = min(4, len(rev_s))
            cols = list(rev_s.index[:take][::-1])          # oldest -> newest dates
            years = [c.year for c in cols]

            def vals(row, absolute=False):
                if row is None:
                    return [np.nan] * take
                r = row.reindex(cols)                       # align across statements
                out = [float(x) / 1e6 if pd.notna(x) else np.nan for x in r.values]
                return [abs(v) if (absolute and pd.notna(v)) else v for v in out]

            ebit_row = get_row(inc, ["EBIT", "Operating Income", "Operating Income Or Loss"])
            ni_row   = get_row(inc, ["Net Income", "Net Income Common Stockholders",
                                     "Net Income From Continuing Operations"])
            pretax_row = get_row(inc, ["Pretax Income", "Income Before Tax",
                                       "Pretax Income Loss"])
            tax_row    = get_row(inc, ["Tax Provision", "Income Tax Expense",
                                       "Tax Provision Benefit"])
            da_row   = get_row(cf, ["Depreciation And Amortization",
                                    "Depreciation Amortization Depletion", "Depreciation"])
            capex_row = get_row(cf, ["Capital Expenditure", "Capital Expenditures",
                                     "Purchase Of PPE"])

            revenue = vals(rev_row)
            ebit = vals(ebit_row)
            net_income = vals(ni_row)
            dep_amort = vals(da_row)
            capex = vals(capex_row, absolute=True)          # store CapEx as positive

            # effective tax rate from latest pretax & tax (clamped to a sane band)
            pretax = _latest(pretax_row); tax = _latest(tax_row)
            if pd.notna(pretax) and pretax not in (0, np.nan) and pd.notna(tax):
                eff_tax = float(np.clip(abs(tax) / abs(pretax), 0.05, 0.35))
            else:
                eff_tax = 0.21

            total_debt = _latest(get_row(bs, ["Total Debt"]))
            if np.isnan(total_debt):
                ltd = _latest(get_row(bs, ["Long Term Debt", "Long Term Debt And Capital Lease Obligation"]))
                std = _latest(get_row(bs, ["Current Debt", "Short Long Term Debt",
                                           "Current Debt And Capital Lease Obligation"]))
                total_debt = np.nansum([ltd, std])
            total_debt = (total_debt or 0.0) / 1e6

            cash = _latest(get_row(bs, ["Cash Cash Equivalents And Short Term Investments",
                                        "Cash And Cash Equivalents", "Cash And Short Term Investments"]))
            cash = (cash if pd.notna(cash) else 0.0) / 1e6

            equity = _latest(get_row(bs, ["Stockholders Equity", "Total Stockholder Equity",
                                          "Common Stock Equity"]))
            equity = (equity if pd.notna(equity) else np.nan) / 1e6

            # ---- market data (fast_info is more reliable than .info) ----
            info = {}
            try: info = tk.info or {}
            except Exception: info = {}
            fi = getattr(tk, "fast_info", None)
            def fi_get(attr):
                try: return getattr(fi, attr)
                except Exception: return None

            price = info.get("currentPrice") or info.get("regularMarketPrice") or fi_get("last_price")
            shares = info.get("sharesOutstanding") or fi_get("shares")
            mktcap = info.get("marketCap") or fi_get("market_cap")
            beta = info.get("beta")
            name = info.get("longName") or info.get("shortName") or ticker
            currency = info.get("currency") or "USD"

            price = float(price) if price else float(fallback["price"])
            shares = (float(shares) / 1e6) if shares else (mktcap / 1e6 / price if mktcap else fallback["shares_out"])
            mktcap = (float(mktcap) / 1e6) if mktcap else price * shares
            beta = float(beta) if beta else 1.10

            return {
                "ticker": ticker.upper(), "name": name, "price": price,
                "shares_out": shares, "beta": beta, "market_cap": mktcap,
                "currency": currency, "years": years, "revenue": revenue,
                "ebit": ebit, "net_income": net_income, "dep_amort": dep_amort,
                "capex": capex, "total_debt": total_debt, "cash": cash,
                "total_equity": equity, "effective_tax": eff_tax, "is_fallback": False,
            }
        except Exception as e:
            last_err = e
            time.sleep(1.5)

    print(f"[!] Live pull failed ({last_err}). Using bundled fallback snapshot.")
    fb = dict(fallback)
    return fb

In [ ]:
# =====================================================================
# CELL 5 — Run Data Collection & Clean Snapshot
# =====================================================================
FIN = fetch_financials(TICKER, force_fallback=FORCE_FALLBACK)

cur = FIN["currency"]
banner = "  *** FALLBACK / ILLUSTRATIVE DATA ***" if FIN.get("is_fallback") else ""
print("=" * 64)
print(f"  {FIN['name']}  ({FIN['ticker']}){banner}")
print("=" * 64)
print(f"  Price                 {cur} {FIN['price']:,.2f}")
print(f"  Shares outstanding    {FIN['shares_out']:,.0f} M")
print(f"  Market cap            {cur} {FIN['market_cap']:,.0f} M")
print(f"  Beta                  {FIN['beta']:.2f}")
print(f"  Total debt            {cur} {FIN['total_debt']:,.0f} M")
print(f"  Cash & ST inv.        {cur} {FIN['cash']:,.0f} M")
print(f"  Net debt              {cur} {FIN['total_debt'] - FIN['cash']:,.0f} M")
print(f"  Effective tax rate    {FIN['effective_tax']:.1%}")
print(f"  Fiscal years pulled   {FIN['years']}")
if FIN.get("is_fallback"):
    print("\n  NOTE: live data was unavailable, so figures above are an "
          "illustrative\n  snapshot. Re-run later or set a different ticker for live data.")

In [ ]:
# =====================================================================
# CELL 6 — Historical Financial Analysis & Feature Engineering
# =====================================================================
def historical_metrics(fin):
    yrs = fin["years"]
    rev = np.array(fin["revenue"], float)
    ebit = np.array(fin["ebit"], float)
    ni = np.array(fin["net_income"], float)
    da = np.array(fin["dep_amort"], float)
    capex = np.array(fin["capex"], float)

    rev_growth = np.concatenate([[np.nan], rev[1:] / rev[:-1] - 1])
    ebit_margin = ebit / rev
    net_margin = ni / rev
    fcf_proxy = ni + da - capex                  # simple FCF proxy (NI + D&A - CapEx)
    fcf_margin = fcf_proxy / rev

    df = pd.DataFrame({
        "Year": yrs,
        "Revenue ($M)": rev,
        "Rev growth %": rev_growth * 100,
        "EBIT ($M)": ebit,
        "EBIT margin %": ebit_margin * 100,
        "Net margin %": net_margin * 100,
        "FCF proxy ($M)": fcf_proxy,
        "FCF margin %": fcf_margin * 100,
    })

    n = len(rev) - 1
    cagr = (rev[-1] / rev[0]) ** (1 / n) - 1 if (rev[0] > 0 and n > 0) else np.nan
    eq = fin.get("total_equity", np.nan)
    nopat_ttm = ebit[-1] * (1 - fin["effective_tax"])
    summary = {
        "Revenue CAGR": cagr,
        "Avg EBIT margin": float(np.mean(ebit_margin)),
        "Avg net margin": float(np.mean(net_margin)),
        "Net debt ($M)": fin["total_debt"] - fin["cash"],
        "Debt / Equity": (fin["total_debt"] / eq) if eq and not np.isnan(eq) else np.nan,
        "ROE (latest)": (ni[-1] / eq) if eq and not np.isnan(eq) else np.nan,
        "ROIC (approx)": (nopat_ttm / (fin["total_debt"] + eq)) if eq and not np.isnan(eq) else np.nan,
    }
    return df, summary

HIST_DF, HIST_SUMMARY = historical_metrics(FIN)
print("HISTORICAL FINANCIALS")
print(HIST_DF.to_string(index=False))
print("\nKEY METRICS")
for k, v in HIST_SUMMARY.items():
    if np.isnan(v):
        print(f"  {k:18s}: n/a")
    elif "$" in k:
        print(f"  {k:18s}: {v:,.0f}")
    elif k in ("Debt / Equity",):
        print(f"  {k:18s}: {v:,.2f}x")
    else:
        print(f"  {k:18s}: {v:.1%}")

In [ ]:
# =====================================================================
# CELL 7 — DCF Assumption Engine
# =====================================================================
# History informs the forward view but does NOT dictate it. Trailing CAGR is
# computed and displayed, but the base-case growth is an explicit analyst
# choice (floored so a single flat year cannot drive the whole forecast).

def build_assumptions(fin, user_overrides=None):
    rev = np.array(fin["revenue"], float)
    ebit = np.array(fin["ebit"], float)
    da = np.array(fin["dep_amort"], float)
    capex = np.array(fin["capex"], float)

    n = len(rev) - 1
    hist_cagr = (rev[-1] / rev[0]) ** (1 / n) - 1 if (rev[0] > 0 and n > 0) else 0.05

    a = {
        "forecast_years": FORECAST_YEARS,
        "rev_growth": round(float(np.clip(hist_cagr, 0.05, 0.10)), 4),
        "ebit_margin": round(float(np.mean(ebit / rev)), 4),
        "tax_rate": round(float(np.clip(fin.get("effective_tax", 0.21), 0.05, 0.35)), 4),
        "da_pct_rev": round(float(np.mean(da / rev)), 4),
        "capex_pct_rev": round(float(np.mean(capex / rev)), 4),
        "nwc_pct_rev": 0.02,
        "risk_free": 0.043,
        "market_risk_premium": 0.047,
        "cost_of_debt": 0.045,
        "terminal_growth": 0.025,
        "exit_multiple": 18.0,
        "_hist_cagr": round(float(hist_cagr), 4),
    }
    if user_overrides:
        a.update(user_overrides)
    return a

ASSUMP = build_assumptions(FIN, USER_OVERRIDES)
print("BASE-CASE ASSUMPTIONS  (edit via USER_OVERRIDES in Cell 3)")
labels = {
    "forecast_years": "Forecast years", "rev_growth": "Revenue growth",
    "ebit_margin": "EBIT margin", "tax_rate": "Tax rate",
    "da_pct_rev": "D&A % revenue", "capex_pct_rev": "CapEx % revenue",
    "nwc_pct_rev": "NWC % revenue", "risk_free": "Risk-free rate",
    "market_risk_premium": "Equity risk premium", "cost_of_debt": "Cost of debt",
    "terminal_growth": "Terminal growth", "exit_multiple": "Exit EV/EBITDA",
}
for k, lab in labels.items():
    v = ASSUMP[k]
    if k == "forecast_years":
        print(f"  {lab:22s}: {v}")
    elif k == "exit_multiple":
        print(f"  {lab:22s}: {v:.1f}x")
    else:
        print(f"  {lab:22s}: {v:.2%}")
print(f"\n  Trailing revenue CAGR (history): {ASSUMP['_hist_cagr']:.2%}  "
      f"-> base-case growth chosen: {ASSUMP['rev_growth']:.2%}")

In [ ]:
# =====================================================================
# CELL 8 — WACC (Weighted Average Cost of Capital)
# =====================================================================
# Cost of equity via CAPM:   Re = Rf + beta * (equity risk premium)
# After-tax cost of debt:    Rd*(1 - tax)
# WACC = (E/V)*Re + (D/V)*Rd*(1 - tax)
#   E = market value of equity, D = total debt, V = E + D

def compute_wacc(fin, a, verbose=True):
    beta = fin.get("beta") or 1.0
    cost_equity = a["risk_free"] + beta * a["market_risk_premium"]
    E, D = fin["market_cap"], fin["total_debt"]
    V = E + D
    we, wd = (E / V, D / V) if V > 0 else (1.0, 0.0)
    after_tax_kd = a["cost_of_debt"] * (1 - a["tax_rate"])
    wacc = we * cost_equity + wd * after_tax_kd
    if verbose:
        print("WACC BUILD-UP")
        print(f"  Beta                         {beta:.2f}")
        print(f"  Risk-free rate               {a['risk_free']:.2%}")
        print(f"  Equity risk premium          {a['market_risk_premium']:.2%}")
        print(f"  Cost of equity (CAPM)        {cost_equity:.2%}")
        print(f"  Pre-tax cost of debt         {a['cost_of_debt']:.2%}")
        print(f"  After-tax cost of debt       {after_tax_kd:.2%}")
        print(f"  Weight of equity (E/V)       {we:.1%}")
        print(f"  Weight of debt   (D/V)       {wd:.1%}")
        print(f"  --------------------------------------")
        print(f"  WACC                         {wacc:.2%}")
    return {"wacc": wacc, "cost_equity": cost_equity, "after_tax_kd": after_tax_kd,
            "we": we, "wd": wd, "E": E, "D": D, "V": V, "beta": beta}

WACC_OUT = compute_wacc(FIN, ASSUMP)
BASE_WACC = WACC_OUT["wacc"]

In [ ]:
# =====================================================================
# CELL 9 — DCF Engine (Free Cash Flow Forecast + dual Terminal Value)
# =====================================================================
# Unlevered Free Cash Flow:
#   UFCF = EBIT*(1-tax) + D&A - CapEx - change in NWC
# Terminal value (two independent methods):
#   Gordon Growth:  TV = UFCF_N*(1+g) / (WACC - g)
#   Exit multiple:  TV = EBITDA_N * (EV/EBITDA multiple)
# Enterprise value = PV(explicit UFCF) + PV(terminal value)
# Equity value      = EV - net debt ;  Intrinsic = Equity / shares

def run_dcf(fin, a, wacc, tv_method="gordon"):
    yrs = a["forecast_years"]
    rev0 = float(fin["revenue"][-1])
    g, tax = a["rev_growth"], a["tax_rate"]

    rows, prev_rev, prev_nwc = [], rev0, rev0 * a["nwc_pct_rev"]
    for t in range(1, yrs + 1):
        rev = prev_rev * (1 + g)
        ebit = rev * a["ebit_margin"]
        nopat = ebit * (1 - tax)
        da = rev * a["da_pct_rev"]
        capex = rev * a["capex_pct_rev"]
        nwc = rev * a["nwc_pct_rev"]
        d_nwc = nwc - prev_nwc
        ufcf = nopat + da - capex - d_nwc
        ebitda = ebit + da
        df = 1 / (1 + wacc) ** t
        rows.append({"Year": t, "Revenue": rev, "EBIT": ebit, "EBITDA": ebitda,
                     "NOPAT": nopat, "D&A": da, "CapEx": capex, "ChgNWC": d_nwc,
                     "UFCF": ufcf, "DiscFactor": df, "PV_UFCF": ufcf * df})
        prev_rev, prev_nwc = rev, nwc

    sched = pd.DataFrame(rows)
    last_df = sched["DiscFactor"].iloc[-1]
    g_term, spread = a["terminal_growth"], wacc - a["terminal_growth"]
    tv_gordon = sched["UFCF"].iloc[-1] * (1 + g_term) / spread if spread > 0 else np.nan
    tv_exit = sched["EBITDA"].iloc[-1] * a["exit_multiple"]
    tv = tv_gordon if tv_method == "gordon" else tv_exit
    pv_tv = tv * last_df

    pv_explicit = sched["PV_UFCF"].sum()
    ev = pv_explicit + pv_tv
    net_debt = fin["total_debt"] - fin["cash"]
    equity_value = ev - net_debt
    shares = fin["shares_out"]
    intrinsic = equity_value / shares if shares else np.nan
    price = fin["price"]
    return {
        "schedule": sched, "wacc": wacc, "tv_method": tv_method,
        "terminal_value": tv, "tv_gordon": tv_gordon, "tv_exit": tv_exit,
        "pv_tv": pv_tv, "pv_explicit": pv_explicit, "enterprise_value": ev,
        "net_debt": net_debt, "equity_value": equity_value, "intrinsic": intrinsic,
        "price": price, "upside": (intrinsic / price - 1) if price else np.nan,
        "margin_of_safety": (intrinsic - price) / intrinsic if intrinsic else np.nan,
        "tv_pct_of_ev": pv_tv / ev if ev else np.nan,
    }

def rating_from_upside(upside, band=RATING_BAND):
    if upside is None or np.isnan(upside):
        return "N/A"
    if upside > band:  return "UNDERVALUED"
    if upside < -band: return "OVERVALUED"
    return "FAIRLY VALUED"

def implied_revenue_growth(fin, a, wacc, lo=-0.10, hi=0.40):
    """Reverse DCF: solve for the constant revenue growth that makes the
    Gordon-Growth intrinsic value equal the current market price."""
    target = fin["price"]
    def f(gv):
        a2 = dict(a); a2["rev_growth"] = gv
        return run_dcf(fin, a2, wacc, "gordon")["intrinsic"] - target
    flo, fhi = f(lo), f(hi)
    if np.isnan(flo) or np.isnan(fhi) or flo * fhi > 0:
        return np.nan
    for _ in range(60):
        mid = (lo + hi) / 2; fm = f(mid)
        if abs(fm) < 1e-6: return mid
        if flo * fm < 0: hi = mid
        else: lo, flo = mid, fm
    return (lo + hi) / 2

print("DCF engine, reverse DCF, and rating functions defined.")

In [ ]:
# =====================================================================
# CELL 10 — Base Case: Forecast Schedule, Terminal Value, Enterprise Value
# =====================================================================
BASE_G = run_dcf(FIN, ASSUMP, BASE_WACC, tv_method="gordon")
BASE_X = run_dcf(FIN, ASSUMP, BASE_WACC, tv_method="exit")

sched = BASE_G["schedule"].copy()
show = sched.copy()
for col in ["Revenue", "EBIT", "EBITDA", "NOPAT", "D&A", "CapEx", "ChgNWC", "UFCF", "PV_UFCF"]:
    show[col] = show[col].map(lambda x: f"{x:,.0f}")
show["DiscFactor"] = sched["DiscFactor"].map(lambda x: f"{x:.3f}")
print(f"5-YEAR UNLEVERED FCF FORECAST  ({cur} M)")
print(show.to_string(index=False))

print(f"\nTERMINAL VALUE & ENTERPRISE VALUE  ({cur} M)")
print(f"  PV of explicit UFCF (yrs 1-{ASSUMP['forecast_years']})   {BASE_G['pv_explicit']:>14,.0f}")
print(f"  Terminal value — Gordon Growth        {BASE_G['tv_gordon']:>14,.0f}")
print(f"  Terminal value — Exit {ASSUMP['exit_multiple']:.0f}x EBITDA       {BASE_X['tv_exit']:>14,.0f}")
print(f"  PV of terminal value (Gordon)         {BASE_G['pv_tv']:>14,.0f}  "
      f"({BASE_G['tv_pct_of_ev']:.0%} of EV)")
print(f"  Enterprise value (Gordon)             {BASE_G['enterprise_value']:>14,.0f}")
print(f"  Enterprise value (Exit multiple)      {BASE_X['enterprise_value']:>14,.0f}")
print(f"  Less: net debt                        {BASE_G['net_debt']:>14,.0f}")

In [ ]:
# =====================================================================
# CELL 11 — Equity Value, Intrinsic Price, Upside & Reverse DCF
# =====================================================================
IMPLIED_G = implied_revenue_growth(FIN, ASSUMP, BASE_WACC)
blended = (BLEND_WEIGHT_GORDON * BASE_G["intrinsic"]
           + (1 - BLEND_WEIGHT_GORDON) * BASE_X["intrinsic"])
blended_upside = blended / FIN["price"] - 1

print(f"INTRINSIC VALUE PER SHARE  ({cur})")
print(f"  Current market price                  {FIN['price']:>10,.2f}")
print(f"  Intrinsic — Gordon Growth             {BASE_G['intrinsic']:>10,.2f}   "
      f"({BASE_G['upside']:+.1%})  {rating_from_upside(BASE_G['upside'])}")
print(f"  Intrinsic — Exit {ASSUMP['exit_multiple']:.0f}x EBITDA           "
      f"{BASE_X['intrinsic']:>10,.2f}   ({BASE_X['upside']:+.1%})  "
      f"{rating_from_upside(BASE_X['upside'])}")
print(f"  Blended fair value ({BLEND_WEIGHT_GORDON:.0%}/{1-BLEND_WEIGHT_GORDON:.0%})         "
      f"{blended:>10,.2f}   ({blended_upside:+.1%})  {rating_from_upside(blended_upside)}")
print(f"  Margin of safety (Gordon)             {BASE_G['margin_of_safety']:>10.1%}")
print("\nREVERSE DCF — what is the market pricing in?")
if np.isnan(IMPLIED_G):
    print("  Implied revenue growth: outside the search range (-10% to +40%).")
else:
    print(f"  To justify today's price on the Gordon-Growth model, revenue must")
    print(f"  grow ~{IMPLIED_G:.1%}/yr for {ASSUMP['forecast_years']} years "
          f"(vs {ASSUMP['_hist_cagr']:.1%} trailing CAGR).")

In [ ]:
# =====================================================================
# CELL 12 — Bull / Base / Bear Scenario Analysis
# =====================================================================
def run_scenario(name, deltas):
    a = dict(ASSUMP)
    a["rev_growth"] = max(0.0, a["rev_growth"] + deltas["rev_growth"])
    a["ebit_margin"] = max(0.01, a["ebit_margin"] + deltas["ebit_margin"])
    a["terminal_growth"] = a["terminal_growth"] + deltas["terminal_growth"]
    a["exit_multiple"] = max(4.0, a["exit_multiple"] + deltas["exit_multiple"])
    wacc = max(0.03, BASE_WACC + deltas["wacc"])
    g = run_dcf(FIN, a, wacc, "gordon")
    x = run_dcf(FIN, a, wacc, "exit")
    blend = BLEND_WEIGHT_GORDON * g["intrinsic"] + (1 - BLEND_WEIGHT_GORDON) * x["intrinsic"]
    return {"name": name, "wacc": wacc, "rev_growth": a["rev_growth"],
            "ebit_margin": a["ebit_margin"], "terminal_growth": a["terminal_growth"],
            "gordon": g["intrinsic"], "exit": x["intrinsic"], "blended": blend,
            "upside": blend / FIN["price"] - 1, "rating": rating_from_upside(blend / FIN["price"] - 1)}

SCEN = {n: run_scenario(n, d) for n, d in SCENARIO_DELTAS.items()}
scen_tbl = pd.DataFrame([{
    "Scenario": s["name"],
    "Rev growth": f"{s['rev_growth']:.1%}",
    "EBIT margin": f"{s['ebit_margin']:.1%}",
    "Term g": f"{s['terminal_growth']:.1%}",
    "WACC": f"{s['wacc']:.1%}",
    "Gordon $": f"{s['gordon']:,.2f}",
    "Exit $": f"{s['exit']:,.2f}",
    "Blended $": f"{s['blended']:,.2f}",
    "Upside": f"{s['upside']:+.1%}",
    "Rating": s["rating"],
} for s in SCEN.values()])
print(f"SCENARIO ANALYSIS  (intrinsic value per share, {cur}; price = {FIN['price']:,.2f})")
print(scen_tbl.to_string(index=False))

In [ ]:
# =====================================================================
# CELL 13 — Sensitivity Analysis Tables
# =====================================================================
def sens_grid(row_vals, col_vals, fn):
    df = pd.DataFrame(index=row_vals, columns=col_vals, dtype=float)
    for r in row_vals:
        for c in col_vals:
            df.loc[r, c] = fn(r, c)
    return df

# (A) WACC (rows) x Terminal growth (cols) — Gordon Growth method
wacc_axis = np.round(np.linspace(BASE_WACC - 0.02, BASE_WACC + 0.02, 5), 4)
g_axis = np.round(np.linspace(ASSUMP["terminal_growth"] - 0.01,
                              ASSUMP["terminal_growth"] + 0.01, 5), 4)
SENS_WG = sens_grid(
    wacc_axis, g_axis,
    lambda w, g: run_dcf(FIN, {**ASSUMP, "terminal_growth": g}, w, "gordon")["intrinsic"])
SENS_WG.index = [f"{x:.2%}" for x in wacc_axis]
SENS_WG.columns = [f"{x:.2%}" for x in g_axis]

# (B) Revenue growth (rows) x EBIT margin (cols) — Gordon Growth method
rg_axis = np.round(np.linspace(max(0.0, ASSUMP["rev_growth"] - 0.04),
                               ASSUMP["rev_growth"] + 0.04, 5), 4)
em_axis = np.round(np.linspace(max(0.02, ASSUMP["ebit_margin"] - 0.06),
                               ASSUMP["ebit_margin"] + 0.06, 5), 4)
SENS_RM = sens_grid(
    rg_axis, em_axis,
    lambda r, m: run_dcf(FIN, {**ASSUMP, "rev_growth": r, "ebit_margin": m}, BASE_WACC, "gordon")["intrinsic"])
SENS_RM.index = [f"{x:.1%}" for x in rg_axis]
SENS_RM.columns = [f"{x:.1%}" for x in em_axis]

# (C) WACC (rows) x Exit multiple (cols) — Exit-multiple method
xm_axis = np.round(np.linspace(ASSUMP["exit_multiple"] - 6, ASSUMP["exit_multiple"] + 6, 5), 1)
SENS_WX = sens_grid(
    wacc_axis, xm_axis,
    lambda w, m: run_dcf(FIN, {**ASSUMP, "exit_multiple": m}, w, "exit")["intrinsic"])
SENS_WX.index = [f"{x:.2%}" for x in wacc_axis]
SENS_WX.columns = [f"{x:.1f}x" for x in xm_axis]

print(f"(A) Intrinsic $/share — WACC (rows) x Terminal growth (cols)  [Gordon]")
print(SENS_WG.round(2).to_string())
print(f"\n(B) Intrinsic $/share — Revenue growth (rows) x EBIT margin (cols)  [Gordon]")
print(SENS_RM.round(2).to_string())
print(f"\n(C) Intrinsic $/share — WACC (rows) x Exit EV/EBITDA (cols)  [Exit multiple]")
print(SENS_WX.round(2).to_string())
print(f"\n  (Current price for reference: {cur} {FIN['price']:,.2f})")

In [ ]:
# =====================================================================
# CELL 14 — Professional Visualizations
# =====================================================================
def money_axis(ax):
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
yrs = [str(y) for y in FIN["years"]]

# (1) Historical revenue
ax[0, 0].bar(yrs, np.array(FIN["revenue"]) / 1000, color=NAVY)
ax[0, 0].set_title("Historical Revenue"); ax[0, 0].set_ylabel(f"{cur} B")
money_axis(ax[0, 0])

# (2) Historical EBIT margin
em = np.array(FIN["ebit"]) / np.array(FIN["revenue"]) * 100
ax[0, 1].plot(yrs, em, marker="o", color=GREEN, lw=2)
ax[0, 1].set_title("EBIT Margin Trend"); ax[0, 1].set_ylabel("%")
for x, y in zip(yrs, em):
    ax[0, 1].annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)

# (3) Forecast UFCF
fy = BASE_G["schedule"]["Year"].astype(int).astype(str)
ax[0, 2].bar(fy, BASE_G["schedule"]["UFCF"] / 1000, color=GREEN, alpha=0.85)
ax[0, 2].set_title("Forecast Unlevered FCF"); ax[0, 2].set_ylabel(f"{cur} B")
ax[0, 2].set_xlabel("Forecast year"); money_axis(ax[0, 2])

# (4) Scenario intrinsic vs price
names = list(SCEN.keys())
blended_vals = [SCEN[n]["blended"] for n in names]
colors = [RED, NAVY, GREEN]
ax[1, 0].bar(names, blended_vals, color=colors, alpha=0.9)
ax[1, 0].axhline(FIN["price"], color="black", ls="--", lw=1.5, label=f"Price {FIN['price']:,.0f}")
ax[1, 0].set_title("Intrinsic Value by Scenario (blended)"); ax[1, 0].set_ylabel(f"{cur}/sh")
ax[1, 0].legend()
for i, v in enumerate(blended_vals):
    ax[1, 0].annotate(f"{v:,.0f}", (i, v), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=8)

# (5) Price vs intrinsic (both methods)
methods = ["Gordon", "Exit mult.", "Blended"]
mvals = [BASE_G["intrinsic"], BASE_X["intrinsic"], blended]
ax[1, 1].bar(methods, mvals, color=[NAVY, GOLD, GREY], alpha=0.9)
ax[1, 1].axhline(FIN["price"], color="black", ls="--", lw=1.5, label=f"Price {FIN['price']:,.0f}")
ax[1, 1].set_title("Base-Case Intrinsic vs Price"); ax[1, 1].set_ylabel(f"{cur}/sh"); ax[1, 1].legend()
for i, v in enumerate(mvals):
    ax[1, 1].annotate(f"{v:,.0f}", (i, v), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=8)

# (6) Sensitivity heatmap (WACC x terminal growth)
grid = SENS_WG.astype(float)
if HAS_SNS:
    sns.heatmap(grid, annot=True, fmt=".0f", cmap="RdYlGn", ax=ax[1, 2],
                cbar_kws={"label": f"{cur}/sh"})
else:
    im = ax[1, 2].imshow(grid.values, cmap="RdYlGn", aspect="auto")
    ax[1, 2].set_xticks(range(len(grid.columns))); ax[1, 2].set_xticklabels(grid.columns)
    ax[1, 2].set_yticks(range(len(grid.index))); ax[1, 2].set_yticklabels(grid.index)
    for (i, j), val in np.ndenumerate(grid.values):
        ax[1, 2].text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax[1, 2])
ax[1, 2].set_title("Sensitivity: WACC x Terminal g")
ax[1, 2].set_xlabel("Terminal growth"); ax[1, 2].set_ylabel("WACC")

fig.suptitle(f"{FIN['name']} ({FIN['ticker']}) — Valuation Dashboard"
             + ("   [FALLBACK DATA]" if FIN.get("is_fallback") else ""),
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ---- Football field (valuation range) ------------------------------------
fig2, axf = plt.subplots(figsize=(11, 4.2))
g_lo, g_hi = SCEN["Bear"]["gordon"], SCEN["Bull"]["gordon"]
x_lo, x_hi = SCEN["Bear"]["exit"], SCEN["Bull"]["exit"]
s_lo, s_hi = float(grid.values.min()), float(grid.values.max())
b_lo = min(BASE_G["intrinsic"], BASE_X["intrinsic"])
b_hi = max(BASE_G["intrinsic"], BASE_X["intrinsic"])
ranges = [
    ("DCF — Gordon Growth\n(bear → bull)", g_lo, g_hi, NAVY),
    ("DCF — Exit Multiple\n(bear → bull)", x_lo, x_hi, GOLD),
    ("Sensitivity (WACC x g)", s_lo, s_hi, GREY),
    ("Base case (method range)", b_lo, b_hi, GREEN),
]
for i, (lab, lo, hi, c) in enumerate(ranges):
    axf.barh(i, hi - lo, left=lo, color=c, alpha=0.75, edgecolor="black")
    axf.annotate(f"{lo:,.0f}", (lo, i), xytext=(-4, 0), textcoords="offset points", va="center", ha="right", fontsize=8)
    axf.annotate(f"{hi:,.0f}", (hi, i), xytext=(4, 0), textcoords="offset points", va="center", ha="left", fontsize=8)
axf.axvline(FIN["price"], color="red", ls="--", lw=2, label=f"Current price {FIN['price']:,.0f}")
axf.set_yticks(range(len(ranges))); axf.set_yticklabels([r[0] for r in ranges])
axf.set_xlabel(f"Intrinsic value per share ({cur})")
axf.set_title(f"{FIN['ticker']} — Valuation Football Field", fontweight="bold")
axf.legend(loc="lower right"); axf.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# =====================================================================
# CELL 15 — Investment Memo Generator
# =====================================================================
def build_memo(fin, a, wacc_out, base_g, base_x, blended, scen, implied_g):
    price = fin["price"]
    blended_up = blended / price - 1
    rating = rating_from_upside(blended_up)
    lows = [s["gordon"] for s in scen.values()] + [s["exit"] for s in scen.values()]
    lo, hi = min(lows), max(lows)
    nd = fin["total_debt"] - fin["cash"]
    flag = "  [BASED ON ILLUSTRATIVE FALLBACK DATA]" if fin.get("is_fallback") else ""

    risks = []
    if base_g["tv_pct_of_ev"] > 0.65:
        risks.append(f"Terminal value drives ~{base_g['tv_pct_of_ev']:.0%} of enterprise "
                     "value, so the result is highly sensitive to the terminal assumption.")
    if not np.isnan(implied_g) and implied_g > a["_hist_cagr"] + 0.03:
        risks.append(f"The market is implying ~{implied_g:.0%} revenue growth vs "
                     f"{a['_hist_cagr']:.0%} trailing CAGR; the price embeds optimism the "
                     "explicit forecast does not.")
    risks.append("Single-stage growth and flat margins simplify a real business that has "
                 "product cycles, mix shift, and competitive dynamics.")
    risks.append("WACC is sensitive to beta, the equity risk premium, and the risk-free rate; "
                 "small changes move the valuation materially.")
    if fin.get("is_fallback"):
        risks.append("Live market data was unavailable; figures rely on a static snapshot.")

    L = []
    L.append("=" * 70)
    L.append(f"  INVESTMENT MEMO — {fin['name']} ({fin['ticker']}){flag}")
    L.append("=" * 70)
    L.append("")
    L.append("COMPANY & MARKET SNAPSHOT")
    L.append(f"  Current price ................ {fin['currency']} {price:,.2f}")
    L.append(f"  Market cap ................... {fin['currency']} {fin['market_cap']:,.0f} M")
    L.append(f"  Net debt ..................... {fin['currency']} {nd:,.0f} M")
    L.append(f"  Beta ......................... {fin['beta']:.2f}")
    L.append("")
    L.append("VALUATION SUMMARY")
    L.append(f"  WACC ......................... {wacc_out['wacc']:.2%}")
    L.append(f"  Intrinsic (Gordon Growth) .... {fin['currency']} {base_g['intrinsic']:,.2f}  ({base_g['upside']:+.1%})")
    L.append(f"  Intrinsic (Exit {a['exit_multiple']:.0f}x EBITDA) .. {fin['currency']} {base_x['intrinsic']:,.2f}  ({base_x['upside']:+.1%})")
    L.append(f"  Blended fair value ........... {fin['currency']} {blended:,.2f}  ({blended_up:+.1%})")
    L.append(f"  Scenario range (bear-bull) ... {fin['currency']} {lo:,.2f}  to  {hi:,.2f}")
    if not np.isnan(implied_g):
        L.append(f"  Market-implied rev growth .... ~{implied_g:.1%}/yr (reverse DCF)")
    L.append("")
    L.append("HISTORICAL TRENDS")
    rg = HIST_DF['Rev growth %'].dropna()
    L.append(f"  Revenue CAGR ................. {HIST_SUMMARY['Revenue CAGR']:.1%}" if not np.isnan(HIST_SUMMARY['Revenue CAGR']) else "  Revenue CAGR ................. n/a")
    L.append(f"  Avg EBIT margin .............. {HIST_SUMMARY['Avg EBIT margin']:.1%}")
    L.append(f"  Avg net margin .............. {HIST_SUMMARY['Avg net margin']:.1%}")
    L.append("")
    L.append("KEY ASSUMPTIONS (base case)")
    L.append(f"  Revenue growth {a['rev_growth']:.1%} | EBIT margin {a['ebit_margin']:.1%} | "
             f"tax {a['tax_rate']:.1%}")
    L.append(f"  Terminal growth {a['terminal_growth']:.1%} | Exit multiple {a['exit_multiple']:.0f}x | "
             f"WACC {wacc_out['wacc']:.1%}")
    L.append("")
    L.append("SCENARIOS")
    for s in scen.values():
        L.append(f"  {s['name']:5s}: blended {fin['currency']} {s['blended']:,.2f}  "
                 f"({s['upside']:+.1%})  -> {s['rating']}")
    L.append("")
    L.append("KEY RISKS")
    for r in risks:
        L.append(f"  - {r}")
    L.append("")
    L.append("RECOMMENDATION")
    L.append(f"  >>> {rating} <<<")
    if rating == "UNDERVALUED":
        thesis = ("The blended intrinsic estimate sits meaningfully above the market price, "
                  "implying the market is under-pricing the company's cash generation. The "
                  "thesis depends on the forecast growth and margins holding.")
    elif rating == "OVERVALUED":
        thesis = ("The blended intrinsic estimate sits below the market price. A single-stage "
                  "DCF structurally undervalues capital-return-heavy compounders, and the "
                  "reverse DCF shows the market is pricing in growth well above the historical "
                  "trend. The gap is concentrated in the terminal assumption, which is the "
                  "honest battleground for this name.")
    else:
        thesis = ("Intrinsic value and market price are within the rating band, so the stock "
                  "looks roughly fairly valued on these assumptions. The exit-multiple and "
                  "perpetuity methods bracket the price, suggesting the market's terminal "
                  "assumptions are reasonable.")
    L.append(f"  {thesis}")
    L.append("")
    L.append("=" * 70)
    L.append("  For educational use only. Not investment advice.")
    L.append("=" * 70)
    return "\n".join(L)

MEMO = build_memo(FIN, ASSUMP, WACC_OUT, BASE_G, BASE_X, blended, SCEN, IMPLIED_G)
print(MEMO)

In [ ]:
# =====================================================================
# CELL 16 — Model Output Summary  (one-glance scorecard)
# =====================================================================
summary = {
    "Ticker": FIN["ticker"],
    "Company": FIN["name"],
    "Current price": f"{cur} {FIN['price']:,.2f}",
    "Intrinsic (Gordon)": f"{cur} {BASE_G['intrinsic']:,.2f}",
    "Intrinsic (Exit mult.)": f"{cur} {BASE_X['intrinsic']:,.2f}",
    "Blended fair value": f"{cur} {blended:,.2f}",
    "Upside / downside": f"{blended/FIN['price']-1:+.1%}",
    "Margin of safety": f"{BASE_G['margin_of_safety']:+.1%}",
    "Enterprise value (Gordon)": f"{cur} {BASE_G['enterprise_value']:,.0f} M",
    "Equity value (Gordon)": f"{cur} {BASE_G['equity_value']:,.0f} M",
    "WACC": f"{BASE_WACC:.2%}",
    "Terminal growth": f"{ASSUMP['terminal_growth']:.2%}",
    "Terminal value (Gordon)": f"{cur} {BASE_G['tv_gordon']:,.0f} M",
    "Implied rev growth (reverse DCF)": (f"{IMPLIED_G:.1%}" if not np.isnan(IMPLIED_G) else "n/a"),
    "Investment rating": rating_from_upside(blended / FIN["price"] - 1),
}
print("=" * 56)
print("  MODEL OUTPUT SUMMARY")
print("=" * 56)
for k, v in summary.items():
    print(f"  {k:34s}: {v}")
print("=" * 56)

# ---- Auto-generated resume bullet from live results ----------------------
_dir = "above" if blended > FIN["price"] else "below"
print("\nAUTO-GENERATED RESUME LINE (from this run):")
print(f'  "Built a Python DCF model that valued {FIN["ticker"]} at '
      f'{cur}{blended:,.0f}/share ({blended/FIN["price"]-1:+.0%} vs market), using a '
      f'{BASE_WACC:.1%} WACC, dual terminal-value methods, and a reverse-DCF cross-check."')

## 18. Educational Explanation — How It All Fits Together

**What each section does**
- **Cells 2–5 (data):** install/import, set the control panel, then pull and normalize statements + market data. The `get_row` helper is the workhorse: yfinance row labels change between versions, so it searches a list of candidate names instead of hard-coding one.
- **Cell 6 (history):** turns raw statements into the metrics analysts actually read — revenue growth, EBIT/net/FCF margins, ROE, ROIC, D/E.
- **Cell 7 (assumptions):** derives base-case inputs from history but treats the forward growth rate as an explicit judgment, not a mechanical copy of trailing CAGR.
- **Cell 8 (WACC):** the discount rate, built from CAPM cost of equity and after-tax cost of debt, weighted by capital structure.
- **Cells 9–11 (DCF):** the engine. Forecast unlevered FCF, discount it, add a terminal value (two methods), bridge enterprise value → equity value → intrinsic price, then run a reverse DCF.
- **Cells 12–13 (stress tests):** scenarios and two-variable sensitivity grids.
- **Cell 14 (charts):** dashboard + football field.
- **Cells 15–16 (output):** written memo and one-glance scorecard.

**What the major variables mean**
- **UFCF (unlevered free cash flow):** cash the business generates before financing. `EBIT*(1-tax) + D&A − CapEx − ΔNWC`.
- **WACC:** the blended required return of all capital providers; the rate that converts future cash to today's value.
- **Terminal value:** the value of all cash flows beyond the explicit forecast — usually the majority of the answer.
- **Net debt:** total debt minus cash; the bridge from enterprise value to equity value.
- **Margin of safety:** how far below intrinsic value the price sits; a cushion for being wrong.

**Finance concepts:** time value of money, CAPM, the enterprise-to-equity bridge, perpetuity vs exit-multiple terminal value, and reverse DCF (solving for the market's implied assumptions).

**Python concepts:** functions as the unit of reuse, dictionaries for structured state, pandas DataFrames for tabular math, numpy vectorization, bisection root-finding for the reverse DCF, and defensive error handling with graceful fallbacks.

**Common bugs / weaknesses (be ready to name these):**
- yfinance is rate-limited and its labels drift; the fallback exists precisely for that.
- CapEx is reported negative in the cash-flow statement; the code takes the absolute value.
- Terminal value can explode or go negative if `WACC ≤ terminal growth`; the code guards against it.
- Single-stage growth and flat margins are simplifications; real models fade growth and ramp margins.
- Book equity as a proxy for the market value of debt is a shortcut (debt is usually close to book, so it is acceptable).

**Honest limitations to volunteer in an interview:** a Gordon-Growth DCF structurally undervalues mega-cap compounders; ~70% of the value can sit in the terminal assumption; and beta/ERP choices swing the answer. Naming these *before* you are asked is what separates a strong candidate from a script-reader.

## 19. Resume & Interview Value

**Resume bullet (clean):**
> Built an automated Python equity-valuation model (DCF, WACC/CAPM, scenario and sensitivity analysis) that ingests any public ticker via the Yahoo Finance API and outputs an intrinsic value, valuation football field, and written investment memo.

**Resume bullet (technical):**
> Engineered a modular DCF engine in Python (pandas/numpy) computing unlevered FCF, a CAPM-based WACC, and dual terminal-value methods (Gordon Growth + EV/EBITDA exit multiple); added bull/base/bear scenarios, two-variable sensitivity grids, and a bisection-based reverse DCF that solves for the market's implied growth rate.

**LinkedIn / GitHub description:**
> An end-to-end equity valuation tool in a single Colab notebook. Enter a ticker; it pulls financial statements and market data, builds a five-year unlevered DCF with a CAPM/WACC discount rate, triangulates terminal value with perpetuity-growth and exit-multiple methods, runs scenario and sensitivity analysis, reverse-engineers the growth the market is pricing in, and generates presentation-quality charts and a written investment memo with a final rating. Built to be robust (graceful fallbacks when data is missing) and honest (it explains, rather than hides, why a perpetuity DCF tends to flag mega-cap compounders as expensive).

**30-second interview pitch:**
> I built a Python tool that values any public company. You give it a ticker, it pulls the financials, projects five years of unlevered free cash flow, discounts them at a CAPM-based WACC, and adds a terminal value using both perpetuity growth and an exit multiple. Then it runs bull/base/bear cases, a sensitivity table, and a reverse DCF that tells me what growth the market is already pricing in. It ends with a one-page investment memo and a buy/hold/sell-style rating.

**2-minute technical explanation:**
> The core is a single `run_dcf` function so the logic is auditable and reusable. I pull statements from yfinance, but because its labels drift between versions and it gets rate-limited, I wrote a fuzzy row-lookup helper and a bundled fallback so the notebook always runs. I normalize everything to millions. Unlevered FCF is EBIT times one-minus-tax, plus D&A, minus CapEx, minus the change in net working capital. The discount rate is a CAPM cost of equity and an after-tax cost of debt, weighted by capital structure. For terminal value I compute both Gordon Growth and an EV/EBITDA exit multiple, because roughly 70% of the value sits there and real desks triangulate. Scenarios and sensitivity tables just call the same engine with shifted inputs. The piece I'm proudest of is the reverse DCF: a bisection solver that backs out the constant revenue growth that would justify today's price. On Apple, the perpetuity method screens it as expensive and the reverse DCF shows the market implying low-twenties growth, which is the honest, defensible insight rather than a number I tuned to look right.

## 20. Potential Upgrades (roadmap)

Concrete next steps that would push this from "strong student project" to "this person could do desk work":

1. **SEC EDGAR integration** — pull exact 10-K/10-Q line items via the EDGAR API instead of yfinance, for audit-grade inputs.
2. **Comparable company analysis** — automatically build a peer set and value on EV/EBITDA, P/E, and EV/Sales multiples; add the comps range to the football field.
3. **Precedent transactions** — a transaction-multiples module for an M&A context.
4. **Monte Carlo simulation** — draw growth, margin, and WACC from distributions to produce a probability distribution of intrinsic value rather than a point estimate.
5. **Two-stage / three-stage growth** — explicit high-growth, fade, and steady-state phases instead of a single growth rate.
6. **Automated PDF memo** — render the investment memo to a formatted PDF (reportlab / weasyprint) for a true leave-behind.
7. **Streamlit dashboard** — wrap the engine in an interactive web app with sliders for every assumption.
8. **Multi-company screener** — loop the engine over a watchlist and rank by upside / margin of safety.
9. **LLM-written memo** — feed the model outputs to an API to draft a polished narrative memo automatically.

---
*Disclaimer: This project is for educational and portfolio purposes only. It is not investment advice, and the outputs depend entirely on user-supplied assumptions.*